<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Atrium_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm.notebook import tqdm
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [ ]:
sample_path = next(root.glob("la*.nii.gz"))

In [ ]:
sample_path_label = label / sample_path.name

In [ ]:
sample_path, sample_path_label

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_path_label)

mri = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
mri.shape,mask.shape

In [ ]:
nib.aff2axcodes(data.affine)

In [ ]:
slice_num = 40

plt.figure(figsize=(6,6))
plt.imshow(mri[:,:,slice_num], cmap="bone")

mask_ = np.ma.masked_where(mask[:,:,slice_num] == 0,
                           mask[:,:,slice_num])

plt.imshow(mask_, alpha=0.5, cmap="autumn")

plt.show()

# **Preprocessing and Creating the Dataset**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [ ]:
import torchio as tio
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [ ]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [ ]:
image_path = sorted(root.glob("*.nii.gz"))
label_path = sorted(root.glob("*.nii.gz"))

In [ ]:
def make_subjects(image_path, label_path):
  subjects = []

  for img_path,lbl_path in zip(image_path, label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        label = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subjects = make_subjects(image_path, label_path)

In [ ]:
train_subject,val_subject=train_test_split(all_subjects,test_size=0.15,random_state=13)

In [ ]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.CropOrPad((208,208,80),mask_name="label"),
    tio.ZNormalization(),
    tio.Clamp(out_min=-3, out_max=3),
    tio.RescaleIntensity(0,1),
])

In [ ]:
train_transform=tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0, 1, 2), flip_probability=0.5),
    tio.RandomAffine(scales=0.05, degrees=10, translation=5),
    tio.RandomNoise(mean=0, std=0.02),
])


val_transform=tio.Compose([`
    base_transform
])

In [ ]:
train_dataset = tio.SubjectsDataset(train_subject,transform=train_transform)
val_dataset = tio.SubjectsDataset(val_subject,transform=val_transform)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=8,num_workers=4,shuffle=True,pin_memory=True)
val_loader = DataLoader(val_dataset,batch_size=8,num_workers=4,shuffle=False,pin_memory=True)

In [ ]:
sample = train_dataset[0]
mri_tensor   = sample["mri"]["data"]    # shape: (1, H, W, D)
label_tensor = sample["label"]["data"]

print(f"   MRI   : {mri_tensor.shape}")
print(f"   Label : {label_tensor.shape}")
print(f" MRI value range : [{mri_tensor.min():.3f}, {mri_tensor.max():.3f}]")
print(f"Label unique vals: {label_tensor.unique()}")

In [ ]:
# for i in range(16):
#     sample = train_dataset[i]
#     img = sample["mri"]["data"]
#     print(i, img.min().item(), img.max().item())

In [ ]:
def show_slice(subject, slice_idx=40):
    mri   = subject["mri"]["data"][0]    # remove channel dim → (H,W,D)
    label = subject["label"]["data"][0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    views = {
        "Axial (top-down)"  : (mri[:,:,slice_idx],   label[:,:,slice_idx]),
        "Coronal (front)"   : (mri[:,slice_idx,:],   label[:,slice_idx,:]),
        "Sagittal (side)"   : (mri[slice_idx,:,:],   label[slice_idx,:,:]),
    }
    for ax, (title, (img, msk)) in zip(axes, views.items()):
        ax.imshow(img, cmap="bone")
        masked = np.ma.masked_where(msk == 0, msk)
        ax.imshow(masked, alpha=0.5, cmap="autumn")
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle("MRI Scan + Atrium Mask (red overlay)", fontsize=14)
    plt.tight_layout()
    plt.show()

show_slice(train_dataset[0], slice_idx=40)